{
 "nbformat": 4,
 "nbformat_minor": 0,
 "metadata": {
  "colab": {"provenance": [], "gpuType": "T4"},
  "kernelspec": {"name": "python3", "display_name": "Python 3"},
  "accelerator": "GPU"
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Parte B — Fine-tuning MAE-AST su ESC-50\n",
    "### Replica di Baade et al. (Interspeech 2022) — Table 1\n",
    "\n",
    "**Obiettivo**: verificare che i checkpoint pretrainati degli autori\n",
    "producano risultati vicini al paper su ESC-50 (target: ~90.0%).\n",
    "\n",
    "**Protocollo**: 5-fold cross-validation (standard ufficiale ESC-50).\n",
    "\n",
    "**Hardware richiesto**: GPU T4 (Colab gratuito è sufficiente).\n",
    "\n",
    "---\n",
    "⚠️ **Prima di eseguire**: assicurati di avere su Google Drive:\n",
    "- `mae_ast_patch_converted.pth` (checkpoint Patch convertito)\n",
    "- `mae_ast_frame_converted.pth` (checkpoint Frame convertito)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Cella 0 — Verifica GPU"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "\n",
    "if torch.cuda.is_available():\n",
    "    nome_gpu = torch.cuda.get_device_name(0)\n",
    "    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9\n",
    "    print(f'✅ GPU: {nome_gpu}  ({vram_gb:.1f} GB VRAM)')\n",
    "else:\n",
    "    raise SystemExit('❌ GPU non trovata. Runtime → Change runtime type → T4 GPU')\n",
    "\n",
    "DEVICE = torch.device('cuda')\n",
    "print(f'   Device: {DEVICE}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Cella 1 — Installa dipendenze"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# timm per il ViT, librosa per l'audio\n",
    "# Le altre (torch, torchaudio, pandas) sono già su Colab\n",
    "!pip install timm==0.9.16 librosa==0.10.1 --quiet\n",
    "\n",
    "import timm, librosa, torchaudio, pandas\n",
    "print(f'timm:       {timm.__version__}')\n",
    "print(f'librosa:    {librosa.__version__}')\n",
    "print(f'torchaudio: {torchaudio.__version__}')\n",
    "print(f'torch:      {torch.__version__}')\n",
    "print('✅ Dipendenze pronte')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Cella 2 — Clona la repo e monta Drive"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os, sys\n",
    "\n",
    "# ── Clona la repo del progetto ────────────────────────────────────────\n",
    "REPO_URL = 'https://github.com/alessiopgg/deep_learning_project.git'\n",
    "REPO_DIR = '/content/mae-ast-poggesi'\n",
    "\n",
    "if os.path.exists(REPO_DIR):\n",
    "    print('Repo già presente, aggiorno...')\n",
    "    !cd {REPO_DIR} && git pull --quiet\n",
    "else:\n",
    "    print('Clono la repo...')\n",
    "    !git clone {REPO_URL} {REPO_DIR} --quiet\n",
    "\n",
    "# Aggiunge la repo al path Python\n",
    "# Senza questa riga, 'from src.preprocessing import ...' fallirebbe\n",
    "if REPO_DIR not in sys.path:\n",
    "    sys.path.insert(0, REPO_DIR)\n",
    "\n",
    "print(f'✅ Repo disponibile in: {REPO_DIR}')\n",
    "\n",
    "# ── Monta Google Drive ────────────────────────────────────────────────\n",
    "from google.colab import drive\n",
    "drive.mount('/content/drive')\n",
    "print('✅ Drive montato')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cella 3 — Configura i percorsi\n",
    "\n",
    "⚠️ **Modifica `CHECKPOINT_PATCH` e `CHECKPOINT_FRAME`**\n",
    "con i percorsi reali dei tuoi file su Drive."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ══════════════════════════════════════════════════════════════════\n",
    "# MODIFICA QUESTI PERCORSI con quelli reali su Drive\n",
    "# ══════════════════════════════════════════════════════════════════\n",
    "CHECKPOINT_PATCH = '/content/drive/MyDrive/mae_ast_patch_converted.pth'\n",
    "CHECKPOINT_FRAME = '/content/drive/MyDrive/mae_ast_frame_converted.pth'\n",
    "# ══════════════════════════════════════════════════════════════════\n",
    "\n",
    "RESULTS_DIR = '/content/drive/MyDrive/mae_ast_results'\n",
    "os.makedirs(RESULTS_DIR, exist_ok=True)\n",
    "\n",
    "# Verifica checkpoint\n",
    "for nome, path in [('Patch', CHECKPOINT_PATCH), ('Frame', CHECKPOINT_FRAME)]:\n",
    "    if os.path.exists(path):\n",
    "        mb = os.path.getsize(path) / 1e6\n",
    "        print(f'  ✅ {nome}: {path}  ({mb:.0f} MB)')\n",
    "    else:\n",
    "        print(f'  ❌ {nome}: file non trovato: {path}')\n",
    "        print(f'     Esegui scripts/convert_checkpoint.py in locale')\n",
    "        print(f'     poi carica il .pth su Drive')\n",
    "\n",
    "print(f'\\n  Risultati salvati in: {RESULTS_DIR}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Cella 4 — Scarica ESC-50"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import glob\n",
    "\n",
    "ESC50_DIR = '/content/ESC-50'\n",
    "CSV_PATH  = f'{ESC50_DIR}/meta/esc50.csv'\n",
    "AUDIO_DIR = f'{ESC50_DIR}/audio'\n",
    "\n",
    "if os.path.exists(CSV_PATH):\n",
    "    print('✅ ESC-50 già presente')\n",
    "else:\n",
    "    print('Scarico ESC-50 (~700 MB)...')\n",
    "    !git clone --depth 1 https://github.com/karolpiczak/ESC-50.git {ESC50_DIR} --quiet\n",
    "    print('✅ ESC-50 scaricato')\n",
    "\n",
    "n_file = len(glob.glob(f'{AUDIO_DIR}/*.wav'))\n",
    "print(f'   File audio: {n_file}/2000')\n",
    "print(f'   CSV:        {CSV_PATH}')\n",
    "\n",
    "# Verifica rapida del CSV\n",
    "import pandas as pd\n",
    "df = pd.read_csv(CSV_PATH)\n",
    "print(f'   Classi:     {df[\"category\"].nunique()}')\n",
    "print(f'   Fold:       {sorted(df[\"fold\"].unique())}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cella 5 — Carica la configurazione\n",
    "\n",
    "Legge gli iperparametri da `config/finetune.yaml`.\n",
    "Puoi modificare il YAML nella repo e fare git pull per aggiornare."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import yaml\n",
    "\n",
    "CONFIG_PATH = f'{REPO_DIR}/config/finetune.yaml'\n",
    "\n",
    "with open(CONFIG_PATH) as f:\n",
    "    yaml_config = yaml.safe_load(f)\n",
    "\n",
    "# Costruisce il dizionario config piatto usato da cross_validation_5fold\n",
    "CONFIG = {\n",
    "    'num_classes':        yaml_config['model']['num_classes'],\n",
    "    'drop_rate':          yaml_config['model']['drop_rate'],\n",
    "    'n_epochs':           yaml_config['training']['n_epochs'],\n",
    "    'batch_size':         yaml_config['training']['batch_size'],\n",
    "    'num_workers':        yaml_config['training']['num_workers'],\n",
    "    'val_ogni':           yaml_config['training']['val_ogni'],\n",
    "    'lr_encoder':         yaml_config['optimizer']['lr_encoder'],\n",
    "    'lr_classificatore':  yaml_config['optimizer']['lr_classificatore'],\n",
    "    'lr_min':             yaml_config['optimizer']['lr_min'],\n",
    "    'weight_decay':       yaml_config['optimizer']['weight_decay'],\n",
    "    'label_smoothing':    yaml_config['optimizer']['label_smoothing'],\n",
    "}\n",
    "\n",
    "print('Configurazione caricata:')\n",
    "for k, v in CONFIG.items():\n",
    "    print(f'  {k:<25} = {v}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cella 6 — Test rapido del modello\n",
    "\n",
    "Prima di lanciare la cross-validation completa, verifica che\n",
    "il checkpoint si carichi correttamente e il forward pass funzioni."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.model import MAEASTFineTune\n",
    "\n",
    "print('='*55)\n",
    "print('TEST CARICAMENTO CHECKPOINT PATCH')\n",
    "print('='*55)\n",
    "\n",
    "modello_test = MAEASTFineTune(\n",
    "    checkpoint_path = CHECKPOINT_PATCH,\n",
    "    num_classes     = CONFIG['num_classes'],\n",
    "    drop_rate       = CONFIG['drop_rate'],\n",
    ").to(DEVICE)\n",
    "\n",
    "# Forward pass con batch finto\n",
    "modello_test.eval()\n",
    "x_test = torch.randn(2, 1, 128, 512).to(DEVICE)\n",
    "with torch.no_grad():\n",
    "    out = modello_test(x_test)\n",
    "\n",
    "print(f'\\nForward pass:')\n",
    "print(f'  Input:  {x_test.shape}')\n",
    "print(f'  Output: {out.shape}  (atteso: [2, 50])')\n",
    "\n",
    "assert out.shape == (2, 50), f'Shape errata: {out.shape}'\n",
    "print('\\n✅ Il checkpoint Patch si carica correttamente!')\n",
    "\n",
    "del modello_test\n",
    "torch.cuda.empty_cache()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cella 7A — Cross-validation: modello PATCH\n",
    "\n",
    "**Target paper**: 90.0%  \n",
    "**Tempo stimato**: ~15-20 minuti su T4"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import json\n",
    "from src.train import cross_validation_5fold\n",
    "\n",
    "print('MODELLO: MAE-AST Patch 12L (chunked masking 75%)')\n",
    "print('Target paper: 90.0%')\n",
    "print()\n",
    "\n",
    "risultati_patch = cross_validation_5fold(\n",
    "    checkpoint_path = CHECKPOINT_PATCH,\n",
    "    csv_path        = CSV_PATH,\n",
    "    audio_dir       = AUDIO_DIR,\n",
    "    config          = CONFIG,\n",
    "    device          = DEVICE,\n",
    ")\n",
    "\n",
    "# Salva risultati su Drive\n",
    "output_path = f'{RESULTS_DIR}/partB_patch.json'\n",
    "with open(output_path, 'w') as f:\n",
    "    json.dump({\n",
    "        'modello':    'MAE-AST Patch 12L',\n",
    "        'masking':    'chunked 75%',\n",
    "        'target':     0.900,\n",
    "        'accuracies': risultati_patch['accuracies'],\n",
    "        'media':      risultati_patch['media'],\n",
    "        'std':        risultati_patch['std'],\n",
    "        'config':     CONFIG,\n",
    "    }, f, indent=2)\n",
    "\n",
    "print(f'\\n✅ Risultati salvati: {output_path}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cella 7B — Cross-validation: modello FRAME\n",
    "\n",
    "**Target paper**: 88.9%  \n",
    "**Tempo stimato**: ~15-20 minuti su T4"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print('MODELLO: MAE-AST Frame 12L (random masking 75%)')\n",
    "print('Target paper: 88.9%')\n",
    "print()\n",
    "\n",
    "risultati_frame = cross_validation_5fold(\n",
    "    checkpoint_path = CHECKPOINT_FRAME,\n",
    "    csv_path        = CSV_PATH,\n",
    "    audio_dir       = AUDIO_DIR,\n",
    "    config          = CONFIG,\n",
    "    device          = DEVICE,\n",
    ")\n",
    "\n",
    "output_path = f'{RESULTS_DIR}/partB_frame.json'\n",
    "with open(output_path, 'w') as f:\n",
    "    json.dump({\n",
    "        'modello':    'MAE-AST Frame 12L',\n",
    "        'masking':    'random 75%',\n",
    "        'target':     0.889,\n",
    "        'accuracies': risultati_frame['accuracies'],\n",
    "        'media':      risultati_frame['media'],\n",
    "        'std':        risultati_frame['std'],\n",
    "        'config':     CONFIG,\n",
    "    }, f, indent=2)\n",
    "\n",
    "print(f'\\n✅ Risultati salvati: {output_path}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Cella 8 — Riepilogo e confronto con il paper"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import matplotlib.pyplot as plt\n",
    "import numpy as np\n",
    "\n",
    "# ── Tabella riepilogo ─────────────────────────────────────────────────\n",
    "print('='*60)\n",
    "print('  RIEPILOGO RISULTATI — Parte B')\n",
    "print('='*60)\n",
    "print(f'  {\"Modello\":<25} {\"Nostro\":>10} {\"Paper\":>10} {\"Delta\":>10}')\n",
    "print('  ' + '─'*55)\n",
    "\n",
    "for nome, risultati, target in [\n",
    "    ('MAE-AST Patch 12L', risultati_patch, 0.900),\n",
    "    ('MAE-AST Frame 12L', risultati_frame, 0.889),\n",
    "]:\n",
    "    media = risultati['media']\n",
    "    std   = risultati['std']\n",
    "    delta = (media - target) * 100\n",
    "    segno = '+' if delta >= 0 else ''\n",
    "    print(f'  {nome:<25} '\n",
    "          f'{media*100:.2f}±{std*100:.2f}% '\n",
    "          f'{target*100:.1f}% '\n",
    "          f'{segno}{delta:.2f}%')\n",
    "\n",
    "print('='*60)\n",
    "\n",
    "# ── Grafici ───────────────────────────────────────────────────────────\n",
    "fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n",
    "\n",
    "# Grafico 1: accuracy per fold\n",
    "ax = axes[0]\n",
    "x     = np.arange(5)\n",
    "width = 0.35\n",
    "\n",
    "bars_p = ax.bar(x - width/2,\n",
    "                [a*100 for a in risultati_patch['accuracies']],\n",
    "                width, label='Patch', color='#2E75B6', alpha=0.85)\n",
    "bars_f = ax.bar(x + width/2,\n",
    "                [a*100 for a in risultati_frame['accuracies']],\n",
    "                width, label='Frame', color='#70AD47', alpha=0.85)\n",
    "\n",
    "ax.axhline(y=90.0, color='#2E75B6', linestyle='--',\n",
    "           linewidth=1.5, alpha=0.6, label='Paper Patch 90.0%')\n",
    "ax.axhline(y=88.9, color='#70AD47', linestyle='--',\n",
    "           linewidth=1.5, alpha=0.6, label='Paper Frame 88.9%')\n",
    "\n",
    "ax.set_xticks(x)\n",
    "ax.set_xticklabels([f'Fold {i+1}' for i in range(5)])\n",
    "ax.set_ylabel('Accuracy (%)')\n",
    "ax.set_ylim(70, 100)\n",
    "ax.set_title('Accuracy per fold — ESC-50')\n",
    "ax.legend(fontsize=8)\n",
    "ax.grid(axis='y', alpha=0.3)\n",
    "\n",
    "# Grafico 2: confronto media con paper\n",
    "ax2 = axes[1]\n",
    "modelli   = ['Paper\\nPatch', 'Nostro\\nPatch', 'Paper\\nFrame', 'Nostro\\nFrame']\n",
    "valori    = [90.0,\n",
    "             risultati_patch['media']*100,\n",
    "             88.9,\n",
    "             risultati_frame['media']*100]\n",
    "colori    = ['#2E75B6', '#4472C4', '#70AD47', '#92D050']\n",
    "\n",
    "bars = ax2.bar(modelli, valori, color=colori, alpha=0.85, edgecolor='white')\n",
    "for bar, val in zip(bars, valori):\n",
    "    ax2.text(bar.get_x() + bar.get_width()/2,\n",
    "             bar.get_height() + 0.3,\n",
    "             f'{val:.1f}%',\n",
    "             ha='center', va='bottom', fontsize=10)\n",
    "\n",
    "ax2.set_ylim(75, 100)\n",
    "ax2.set_ylabel('Accuracy (%)')\n",
    "ax2.set_title('Confronto con il paper')\n",
    "ax2.grid(axis='y', alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "\n",
    "# Salva grafico su Drive\n",
    "graph_path = f'{RESULTS_DIR}/partB_risultati.png'\n",
    "plt.savefig(graph_path, dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "print(f'\\n✅ Grafico salvato: {graph_path}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Note sulla riproducibilità\n",
    "\n",
    "È normale che i risultati differiscano leggermente dal paper (±2%).\n",
    "Le fonti di scarto sono:\n",
    "\n",
    "1. **Conversione checkpoint** — il mapping fairseq→timm potrebbe\n",
    "   non essere perfetto al 100%. Il numero di chiavi caricate\n",
    "   correttamente è documentato nell'output della Cella 6.\n",
    "\n",
    "2. **Iperparametri di fine-tuning** — il paper non pubblica tutti\n",
    "   i dettagli. Usiamo i valori della libreria SSAST pubblica.\n",
    "\n",
    "3. **Seed casuale** — non fissato per semplicità. La variabilità\n",
    "   è catturata dalla deviazione standard dei 5 fold.\n",
    "\n",
    "**Criteri di successo**:\n",
    "- Patch > 88%: conversione riuscita, pipeline corretto\n",
    "- Patch < 85%: problema nella conversione da investigare"
   ]
  }
 ]
}